# Primer experimento de ML clásico con MLflow
Este notebook entrena una regresión logística con Iris y registra parámetros, métricas y el modelo. Requiere el servidor local descrito en `README.md`.

In [ ]:
import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature
from sklearn import datasets
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split

mlflow.set_tracking_uri("http://127.0.0.1:5001")
mlflow.set_experiment("classic-ml-iris-notebook")

In [ ]:
iris = datasets.load_iris(as_frame=True)
X_train, X_test, y_train, y_test = train_test_split(
    iris.data, iris.target, test_size=0.2, random_state=42, stratify=iris.target
)
params = {"solver": "lbfgs", "max_iter": 1000, "random_state": 42}
model = LogisticRegression(**params)

In [ ]:
with mlflow.start_run(run_name="notebook-logistic-regression"):
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    metrics = {
        "accuracy": accuracy_score(y_test, predictions),
        "f1_weighted": f1_score(y_test, predictions, average="weighted"),
    }
    mlflow.log_params(params)
    mlflow.log_metrics(metrics)
    model_info = mlflow.sklearn.log_model(
        sk_model=model,
        name="iris_model",
        signature=infer_signature(X_train, model.predict(X_train)),
        input_example=X_train.head(3),
    )
metrics, model_info.model_uri

In [ ]:
loaded_model = mlflow.pyfunc.load_model(model_info.model_uri)
sample = X_test.head(4)
result = sample.copy()
result["actual_class"] = y_test.loc[sample.index]
result["predicted_class"] = loaded_model.predict(sample).astype(int)
result